In [1]:
import random
from datetime import datetime, timedelta
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

# --- Настройки ---
NUM_PAGES = 10
ROWS_PER_PAGE = 100
OUTPUT_FILE = "transactions.xlsx"

CONTRACTORS = [
    "ООО Ромашка", "АО ТехноСервис", "ИП Иванов И.И.", "ООО СтройГрад",
    "ПАО ЭнергоСбыт", "ООО ЛогистикПлюс", "ЗАО АльфаТрейд", "ИП Петров П.П.",
    "ООО МедТехника", "АО ПромРесурс", "ООО ФинКонсалт", "ИП Сидоров С.С.",
    "ООО АгроПром", "АО ГазТранс", "ООО МеталлИнвест", "ИП Кузнецов К.К.",
    "ООО НефтеСервис", "АО СтройМонтаж", "ООО ТоргСнаб", "ИП Смирнова А.А.",
]

RISK_CATEGORIES = ["низкий", "средний", "высокий", "критический"]
OPERATION_TYPES = ["поступление", "списание"]

# Цвета для подсветки категорий риска
RISK_FILLS = {
    "низкий":      PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid"),
    "средний":     PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid"),
    "высокий":     PatternFill(start_color="FFCC99", end_color="FFCC99", fill_type="solid"),
    "критический": PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid"),
}

# Цвета для подсветки типа операции
TYPE_FILLS = {
    "поступление": PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid"),
    "списание":    PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid"),
}

HEADERS = ["id", "date", "contractor", "amount", "risk", "type"]
COLUMN_WIDTHS = [8, 14, 25, 16, 14, 16]

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2025, 12, 31)


def random_date() -> str:
    delta = END_DATE - START_DATE
    return (START_DATE + timedelta(days=random.randint(0, delta.days))).strftime("%Y-%m-%d")


def style_header(ws) -> None:
    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="305496", end_color="305496", fill_type="solid")
    center = Alignment(horizontal="center", vertical="center")

    for col_idx, (name, width) in enumerate(zip(HEADERS, COLUMN_WIDTHS), start=1):
        cell = ws.cell(row=1, column=col_idx)
        cell.value = name
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = center
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.freeze_panes = "A2"  # фиксируем заголовок


def main() -> None:
    random.seed(42)  # для воспроизводимости

    wb = Workbook()
    wb.remove(wb.active)  # убираем дефолтный лист "Sheet"

    transaction_id = 1

    for page in range(1, NUM_PAGES + 1):
        ws = wb.create_sheet(title=f"Страница {page}")
        style_header(ws)

        for row in range(2, ROWS_PER_PAGE + 2):
            risk = random.choice(RISK_CATEGORIES)
            op_type = random.choice(OPERATION_TYPES)

            ws.cell(row=row, column=1, value=transaction_id)
            ws.cell(row=row, column=2, value=random_date())
            ws.cell(row=row, column=3, value=random.choice(CONTRACTORS))

            amount_cell = ws.cell(row=row, column=4, value=round(random.uniform(1_000, 10_000_000), 2))
            amount_cell.number_format = "# ##0.00"

            risk_cell = ws.cell(row=row, column=5, value=risk)
            risk_cell.fill = RISK_FILLS[risk]

            type_cell = ws.cell(row=row, column=6, value=op_type)
            type_cell.fill = TYPE_FILLS[op_type]

            transaction_id += 1

    wb.save(OUTPUT_FILE)
    print(f"Файл '{OUTPUT_FILE}' создан: {NUM_PAGES} листов × {ROWS_PER_PAGE} транзакций "
          f"= {NUM_PAGES * ROWS_PER_PAGE} записей (столбцов: {len(HEADERS)}).")


if __name__ == "__main__":
    main()

Файл 'transactions.xlsx' создан: 10 листов × 100 транзакций = 1000 записей (столбцов: 6).


In [8]:
"""
Скрипт решает задания 2.1–2.10 ОТДЕЛЬНО для каждого варианта (листа)
из файла transactions.xlsx и сохраняет результаты в report.docx.

Каждый вариант = один лист Excel = 100 транзакций.
Вариант N обрабатывается независимо от остальных.

Установка зависимостей:
    pip install pandas openpyxl python-docx
"""

import io

import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH


INPUT_FILE = "transactions.xlsx"
OUTPUT_FILE = "report.docx"

RISK_ORDER = ["низкий", "средний", "высокий", "критический"]


# ---------- Вспомогательные функции для отчёта ----------

def add_heading(doc: Document, text: str, level: int = 1) -> None:
    h = doc.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x3B, 0x73)


def add_paragraph(doc: Document, text: str, bold: bool = False) -> None:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold


def add_code_block(doc: Document, text: str) -> None:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.name = "Consolas"
    run.font.size = Pt(9)
    p.paragraph_format.space_after = Pt(6)


def add_dataframe(doc: Document, df: pd.DataFrame, index: bool = True) -> None:
    add_code_block(doc, df.to_string(index=index))


def fmt_money(x: float) -> str:
    return f"{x:,.2f} руб.".replace(",", " ")


# ---------- Решение заданий 2.1–2.10 для одного варианта ----------

def solve_variant(doc: Document, df: pd.DataFrame, variant_name: str) -> None:
    """Решает все задания для одного листа/варианта и пишет в doc."""

    # Заголовок варианта (новая страница)
    doc.add_page_break()
    add_heading(doc, f"ВАРИАНТ: {variant_name}", level=0)

    # ============================================================
    # Задание 2.1
    # ============================================================
    add_heading(doc, "Задание 2.1. Загрузка данных", level=2)

    add_paragraph(doc, f"Источник: лист «{variant_name}» файла {INPUT_FILE}")
    add_paragraph(doc, f"Форма (shape): {df.shape}", bold=True)

    add_paragraph(doc, "Список столбцов:")
    add_paragraph(doc, ", ".join(df.columns.astype(str)))

    add_paragraph(doc, "df.info():")
    buf = io.StringIO()
    df.info(buf=buf)
    add_code_block(doc, buf.getvalue())

    add_paragraph(doc, "df.describe():")
    add_dataframe(doc, df.describe())

    # ============================================================
    # Задание 2.2
    # ============================================================
    add_heading(doc, "Задание 2.2. Всего транзакций и уникальных контрагентов", level=2)

    add_paragraph(doc, f"Всего транзакций: {len(df)}")
    add_paragraph(doc, f"Уникальных контрагентов: {df['contractor'].nunique()}")

    # ============================================================
    # Задание 2.3
    # ============================================================
    add_heading(doc, "Задание 2.3. Статистика по сумме", level=2)

    add_paragraph(doc, f"Общая сумма: {fmt_money(df['amount'].sum())}")
    add_paragraph(doc, f"Среднее: {fmt_money(df['amount'].mean())}")
    add_paragraph(doc, f"Медиана: {fmt_money(df['amount'].median())}")
    add_paragraph(doc, f"Минимум: {fmt_money(df['amount'].min())}")
    add_paragraph(doc, f"Максимум: {fmt_money(df['amount'].max())}")

    # ============================================================
    # Задание 2.4
    # ============================================================
    add_heading(doc, "Задание 2.4. Крупные транзакции (> 70 % от max)", level=2)

    large = df[df["amount"] > 0.7 * df["amount"].max()]
    add_paragraph(doc, f"Количество крупных транзакций: {len(large)}")
    if len(large):
        add_paragraph(doc, "Первые 5 крупных транзакций:")
        add_dataframe(doc, large.head())

    # ============================================================
    # Задание 2.5
    # ============================================================
    add_heading(doc, "Задание 2.5. Высокий и критический риск", level=2)

    high_risk = df[df["risk"].isin(["высокий", "критический"])]
    add_paragraph(doc, f"Количество транзакций: {len(high_risk)}")
    add_paragraph(doc, f"Общая сумма: {fmt_money(high_risk['amount'].sum())}")
    if len(high_risk):
        add_paragraph(doc, "Разбивка по категориям:")
        add_dataframe(doc, high_risk["risk"].value_counts().rename("count").to_frame())

    # ============================================================
    # Задание 2.6
    # ============================================================
    add_heading(doc, "Задание 2.6. Статистика по контрагентам", level=2)

    by_contractor = (
        df.groupby("contractor")["amount"]
          .agg(count="count", total="sum", mean="mean", median="median", max="max")
          .sort_values("total", ascending=False)
    )
    add_paragraph(doc, "Результат (отсортировано по убыванию суммы):")
    add_dataframe(doc, by_contractor)

    # ============================================================
    # Задание 2.7
    # ============================================================
    add_heading(doc, "Задание 2.7. Статистика по типу операции", level=2)

    by_type = df.groupby("type")["amount"].agg(total="sum", mean="mean")
    add_paragraph(doc, "Сумма и среднее по каждой группе:")
    add_dataframe(doc, by_type)

    # ============================================================
    # Задание 2.8
    # ============================================================
    add_heading(doc, "Задание 2.8. Сводная таблица (контрагенты × риск)", level=2)

    pivot = pd.pivot_table(
        df,
        index="contractor",
        columns="risk",
        values="amount",
        aggfunc="sum",
        fill_value=0,
    )
    pivot = pivot[[c for c in RISK_ORDER if c in pivot.columns]]
    add_paragraph(doc, "Суммы по контрагентам и категориям риска:")
    add_dataframe(doc, pivot)

    # ============================================================
    # Задание 2.9
    # ============================================================
    add_heading(doc, "Задание 2.9. Подозрительные транзакции (is_suspicious)", level=2)

    df = df.copy()
    df["is_suspicious"] = (df["amount"] > 0.9 * df["amount"].max()) | (df["risk"] == "критический")
    add_paragraph(doc, "Столбец is_suspicious создан.")
    add_paragraph(doc, f"Количество подозрительных транзакций: {int(df['is_suspicious'].sum())}")
    suspicious = df[df["is_suspicious"]]
    if len(suspicious):
        add_paragraph(doc, "Первые 5 подозрительных транзакций:")
        add_dataframe(doc, suspicious.head())

    # ============================================================
    # Задание 2.10
    # ============================================================
    add_heading(doc, "Задание 2.10. amount_k и категория суммы", level=2)

    df["amount_k"] = df["amount"] / 1000

    def categorize(amount_k: float) -> str:
        if amount_k < 0.3 * df["amount_k"].max():
            return "мелкая"
        elif amount_k < 0.7 * df["amount_k"].max():
            return "средняя"
        return "крупная"

    df["category"] = df["amount_k"].apply(categorize)

    add_paragraph(doc, "Столбцы amount_k и category созданы.")
    add_paragraph(doc, "Пример первых 10 строк:")
    add_dataframe(doc, df[["id", "amount", "amount_k", "category"]].head(10))

    add_paragraph(doc, "Распределение по категориям:")
    cat_stats = (
        df.groupby("category")["amount"]
          .agg(count="count", total="sum", mean="mean")
          .sort_values("total", ascending=False)
    )
    add_dataframe(doc, cat_stats)


# ---------- Основной сценарий ----------

def main() -> None:
    doc = Document()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(11)

    title = doc.add_heading("Отчёт по вариантам (листам) файла transactions.xlsx", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Читаем все листы как словарь {имя_листа: DataFrame}
    sheets = pd.read_excel(INPUT_FILE, sheet_name=None, engine="openpyxl")
    sheet_names = list(sheets.keys())

    add_paragraph(doc, f"Файл: {INPUT_FILE}")
    add_paragraph(doc, f"Всего вариантов (листов): {len(sheet_names)}")
    add_paragraph(doc, "Список вариантов:")
    for name in sheet_names:
        add_paragraph(doc, f"  • {name} — {len(sheets[name])} строк")

    # Обрабатываем каждый вариант отдельно
    for name in sheet_names:
        df_variant = sheets[name].copy()
        solve_variant(doc, df_variant, variant_name=name)

    doc.save(OUTPUT_FILE)
    print(f"Отчёт сохранён в '{OUTPUT_FILE}'")
    print(f"Обработано вариантов: {len(sheet_names)}")


if __name__ == "__main__":
    main()

Отчёт сохранён в 'report.docx'
Обработано вариантов: 10


In [ ]:
"""
Скрипт решает задания 2.1–2.10 ОТДЕЛЬНО для каждого варианта (листа)
из файла transactions.xlsx и сохраняет результаты в report.docx.

Каждый вариант = один лист Excel = 100 транзакций.
Вариант N обрабатывается независимо от остальных.

Установка зависимостей:
    pip install pandas openpyxl python-docx
"""

import io

import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH


INPUT_FILE = "transactions.xlsx"
OUTPUT_FILE = "report.docx"

RISK_ORDER = ["низкий", "средний", "высокий", "критический"]


# ---------- Вспомогательные функции для отчёта ----------

def add_heading(doc: Document, text: str, level: int = 1) -> None:
    h = doc.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x3B, 0x73)


def add_paragraph(doc: Document, text: str, bold: bool = False) -> None:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold


def add_code_block(doc: Document, text: str) -> None:
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.name = "Consolas"
    run.font.size = Pt(9)
    p.paragraph_format.space_after = Pt(6)


def add_dataframe(doc: Document, df: pd.DataFrame, index: bool = True) -> None:
    add_code_block(doc, df.to_string(index=index))


def fmt_money(x: float) -> str:
    return f"{x:,.2f} руб.".replace(",", " ")


# ---------- Решение заданий 2.1–2.10 для одного варианта ----------

def solve_variant(doc: Document, df: pd.DataFrame, variant_name: str) -> None:
    """Решает все задания для одного листа/варианта и пишет в doc."""

    # Заголовок варианта (новая страница)
    doc.add_page_break()
    add_heading(doc, f"ВАРИАНТ: {variant_name}", level=0)

    # ============================================================
    # Задание 2.1
    # ============================================================
    add_heading(doc, "Задание 2.1. Загрузка данных", level=2)

    add_paragraph(doc, f"Источник: лист «{variant_name}» файла {INPUT_FILE}")
    add_paragraph(doc, f"Форма (shape): {df.shape}", bold=True)

    add_paragraph(doc, "Список столбцов:")
    add_paragraph(doc, ", ".join(df.columns.astype(str)))

    add_paragraph(doc, "df.info():")
    buf = io.StringIO()
    df.info(buf=buf)
    add_code_block(doc, buf.getvalue())

    add_paragraph(doc, "df.describe():")
    add_dataframe(doc, df.describe())

    # ============================================================
    # Задание 2.2
    # ============================================================
    add_heading(doc, "Задание 2.2. Всего транзакций и уникальных контрагентов", level=2)

    add_paragraph(doc, f"Всего транзакций: {len(df)}")
    add_paragraph(doc, f"Уникальных контрагентов: {df['contractor'].nunique()}")

    # ============================================================
    # Задание 2.3
    # ============================================================
    add_heading(doc, "Задание 2.3. Статистика по сумме", level=2)

    add_paragraph(doc, f"Общая сумма: {fmt_money(df['amount'].sum())}")
    add_paragraph(doc, f"Среднее: {fmt_money(df['amount'].mean())}")
    add_paragraph(doc, f"Медиана: {fmt_money(df['amount'].median())}")
    add_paragraph(doc, f"Минимум: {fmt_money(df['amount'].min())}")
    add_paragraph(doc, f"Максимум: {fmt_money(df['amount'].max())}")

    # ============================================================
    # Задание 2.4
    # ============================================================
    add_heading(doc, "Задание 2.4. Крупные транзакции (> 50 000)", level=2)

    large = df[df["amount"] > 50_000]
    add_paragraph(doc, f"Количество крупных транзакций: {len(large)}")
    if len(large):
        add_paragraph(doc, "Первые 5 крупных транзакций:")
        add_dataframe(doc, large.head())

    # ============================================================
    # Задание 2.5
    # ============================================================
    add_heading(doc, "Задание 2.5. Высокий и критический риск", level=2)

    high_risk = df[df["risk"].isin(["высокий", "критический"])]
    add_paragraph(doc, f"Количество транзакций: {len(high_risk)}")
    add_paragraph(doc, f"Общая сумма: {fmt_money(high_risk['amount'].sum())}")
    if len(high_risk):
        add_paragraph(doc, "Разбивка по категориям:")
        add_dataframe(doc, high_risk["risk"].value_counts().rename("count").to_frame())

    # ============================================================
    # Задание 2.6
    # ============================================================
    add_heading(doc, "Задание 2.6. Статистика по контрагентам", level=2)

    by_contractor = (
        df.groupby("contractor")["amount"]
          .agg(count="count", total="sum", mean="mean", median="median", max="max")
          .sort_values("total", ascending=False)
    )
    add_paragraph(doc, "Результат (отсортировано по убыванию суммы):")
    add_dataframe(doc, by_contractor)

    # ============================================================
    # Задание 2.7
    # ============================================================
    add_heading(doc, "Задание 2.7. Статистика по типу операции", level=2)

    by_type = df.groupby("type")["amount"].agg(total="sum", mean="mean")
    add_paragraph(doc, "Сумма и среднее по каждой группе:")
    add_dataframe(doc, by_type)

    # ============================================================
    # Задание 2.8
    # ============================================================
    add_heading(doc, "Задание 2.8. Сводная таблица (контрагенты × риск)", level=2)

    pivot = pd.pivot_table(
        df,
        index="contractor",
        columns="risk",
        values="amount",
        aggfunc="sum",
        fill_value=0,
    )
    pivot = pivot[[c for c in RISK_ORDER if c in pivot.columns]]
    add_paragraph(doc, "Суммы по контрагентам и категориям риска:")
    add_dataframe(doc, pivot)

    # ============================================================
    # Задание 2.9
    # ============================================================
    add_heading(doc, "Задание 2.9. Подозрительные транзакции (is_suspicious)", level=2)

    df = df.copy()
    df["is_suspicious"] = (df["amount"] > 100_000) | (df["risk"] == "критический")
    add_paragraph(doc, "Столбец is_suspicious создан.")
    add_paragraph(doc, f"Количество подозрительных транзакций: {int(df['is_suspicious'].sum())}")
    suspicious = df[df["is_suspicious"]]
    if len(suspicious):
        add_paragraph(doc, "Первые 5 подозрительных транзакций:")
        add_dataframe(doc, suspicious.head())

    # ============================================================
    # Задание 2.10
    # ============================================================
    add_heading(doc, "Задание 2.10. amount_k и категория суммы", level=2)

    df["amount_k"] = df["amount"] / 1000

    def categorize(amount_k: float) -> str:
        if amount_k < 100:
            return "мелкая"
        elif amount_k < 1000:
            return "средняя"
        return "крупная"

    df["category"] = df["amount_k"].apply(categorize)

    add_paragraph(doc, "Столбцы amount_k и category созданы.")
    add_paragraph(doc, "Пример первых 10 строк:")
    add_dataframe(doc, df[["id", "amount", "amount_k", "category"]].head(10))

    add_paragraph(doc, "Распределение по категориям:")
    cat_stats = (
        df.groupby("category")["amount"]
          .agg(count="count", total="sum", mean="mean")
          .sort_values("total", ascending=False)
    )
    add_dataframe(doc, cat_stats)


# ---------- Основной сценарий ----------

def main() -> None:
    doc = Document()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(11)

    title = doc.add_heading("Отчёт по вариантам (листам) файла transactions.xlsx", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Читаем все листы как словарь {имя_листа: DataFrame}
    sheets = pd.read_excel(INPUT_FILE, sheet_name=None, engine="openpyxl")
    sheet_names = list(sheets.keys())

    add_paragraph(doc, f"Файл: {INPUT_FILE}")
    add_paragraph(doc, f"Всего вариантов (листов): {len(sheet_names)}")
    add_paragraph(doc, "Список вариантов:")
    for name in sheet_names:
        add_paragraph(doc, f"  • {name} — {len(sheets[name])} строк")

    # Обрабатываем каждый вариант отдельно
    for name in sheet_names:
        df_variant = sheets[name].copy()
        solve_variant(doc, df_variant, variant_name=name)

    doc.save(OUTPUT_FILE)
    print(f"Отчёт сохранён в '{OUTPUT_FILE}'")
    print(f"Обработано вариантов: {len(sheet_names)}")


if __name__ == "__main__":
    main()

Файл 'transactions.xlsx' создан: 10 листов × 100 транзакций = 1000 записей.


In [7]:
"""
Скрипт решает задания 2.1–2.10 ОТДЕЛЬНО для каждого варианта (листа)
из transactions.xlsx и сохраняет результаты в report.docx.

Установка зависимостей:
    pip install pandas openpyxl python-docx
"""

import io
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

INPUT_FILE = "transactions.xlsx"
OUTPUT_FILE = "report.docx"

RISK_ORDER = ["низкий", "средний", "высокий", "критический"]


# ---------- Вспомогательные функции ----------

def add_heading(doc, text, level=1):
    h = doc.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x3B, 0x73)


def add_paragraph(doc, text, bold=False):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold


def add_code_block(doc, text):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.name = "Consolas"
    run.font.size = Pt(9)
    p.paragraph_format.space_after = Pt(6)


def add_dataframe(doc, df, index=True):
    add_code_block(doc, df.to_string(index=index))


def fmt_money(x):
    return f"{x:,.2f} руб.".replace(",", " ")


# ---------- Решение заданий 2.1–2.10 для одного варианта ----------

def solve_variant(doc, df, variant_name):
    doc.add_page_break()
    add_heading(doc, f"ВАРИАНТ: {variant_name}", level=0)

    # 2.1
    add_heading(doc, "Задание 2.1. Загрузка данных", level=2)
    add_paragraph(doc, f"Источник: лист «{variant_name}» файла {INPUT_FILE}")
    add_paragraph(doc, f"Форма (shape): {df.shape}", bold=True)
    add_paragraph(doc, "Список столбцов:")
    add_paragraph(doc, ", ".join(df.columns.astype(str)))
    add_paragraph(doc, "df.info():")
    buf = io.StringIO()
    df.info(buf=buf)
    add_code_block(doc, buf.getvalue())
    add_paragraph(doc, "df.describe():")
    add_dataframe(doc, df.describe())

    # 2.2
    add_heading(doc, "Задание 2.2. Всего транзакций и уникальных контрагентов", level=2)
    add_paragraph(doc, f"Всего транзакций: {len(df)}")
    add_paragraph(doc, f"Уникальных контрагентов: {df['contractor'].nunique()}")

    # 2.3
    add_heading(doc, "Задание 2.3. Статистика по сумме", level=2)
    add_paragraph(doc, f"Общая сумма: {fmt_money(df['amount'].sum())}")
    add_paragraph(doc, f"Среднее: {fmt_money(df['amount'].mean())}")
    add_paragraph(doc, f"Медиана: {fmt_money(df['amount'].median())}")
    add_paragraph(doc, f"Минимум: {fmt_money(df['amount'].min())}")
    add_paragraph(doc, f"Максимум: {fmt_money(df['amount'].max())}")

    # 2.4
    add_heading(doc, "Задание 2.4. Крупные транзакции (> 50 000)", level=2)
    large = df[df["amount"] > 50_000]
    add_paragraph(doc, f"Количество крупных транзакций: {len(large)}")
    if len(large):
        add_paragraph(doc, "Первые 5 крупных транзакций:")
        add_dataframe(doc, large.head())

    # 2.5
    add_heading(doc, "Задание 2.5. Высокий и критический риск", level=2)
    high_risk = df[df["risk"].isin(["высокий", "критический"])]
    add_paragraph(doc, f"Количество транзакций: {len(high_risk)}")
    add_paragraph(doc, f"Общая сумма: {fmt_money(high_risk['amount'].sum())}")
    if len(high_risk):
        add_paragraph(doc, "Разбивка по категориям:")
        add_dataframe(doc, high_risk["risk"].value_counts().rename("count").to_frame())

    # 2.6
    add_heading(doc, "Задание 2.6. Статистика по контрагентам", level=2)
    by_contractor = (
        df.groupby("contractor")["amount"]
          .agg(count="count", total="sum", mean="mean", median="median", max="max")
          .sort_values("total", ascending=False)
    )
    add_paragraph(doc, "Результат (отсортировано по убыванию суммы):")
    add_dataframe(doc, by_contractor)

    # 2.7
    add_heading(doc, "Задание 2.7. Статистика по типу операции", level=2)
    by_type = df.groupby("type")["amount"].agg(total="sum", mean="mean")
    add_paragraph(doc, "Сумма и среднее по каждой группе:")
    add_dataframe(doc, by_type)

    # 2.8
    add_heading(doc, "Задание 2.8. Сводная таблица (контрагенты × риск)", level=2)
    pivot = pd.pivot_table(
        df, index="contractor", columns="risk",
        values="amount", aggfunc="sum", fill_value=0,
    )
    pivot = pivot[[c for c in RISK_ORDER if c in pivot.columns]]
    add_paragraph(doc, "Суммы по контрагентам и категориям риска:")
    add_dataframe(doc, pivot)

    # 2.9
    add_heading(doc, "Задание 2.9. Подозрительные транзакции (is_suspicious)", level=2)
    df = df.copy()
    df["is_suspicious"] = (df["amount"] > 100_000) | (df["risk"] == "критический")
    add_paragraph(doc, "Столбец is_suspicious создан.")
    add_paragraph(doc, f"Количество подозрительных транзакций: {int(df['is_suspicious'].sum())}")
    suspicious = df[df["is_suspicious"]]
    if len(suspicious):
        add_paragraph(doc, "Первые 5 подозрительных транзакций:")
        add_dataframe(doc, suspicious.head())

    # 2.10
    add_heading(doc, "Задание 2.10. amount_k и категория суммы", level=2)
    df["amount_k"] = df["amount"] / 1000

    def categorize(amount_k):
        if amount_k < 100:
            return "мелкая"
        elif amount_k < 1000:
            return "средняя"
        return "крупная"

    df["category"] = df["amount_k"].apply(categorize)
    add_paragraph(doc, "Столбцы amount_k и category созданы.")
    add_paragraph(doc, "Пример первых 10 строк:")
    add_dataframe(doc, df[["id", "amount", "amount_k", "category"]].head(10))
    add_paragraph(doc, "Распределение по категориям:")
    cat_stats = (
        df.groupby("category")["amount"]
          .agg(count="count", total="sum", mean="mean")
          .sort_values("total", ascending=False)
    )
    add_dataframe(doc, cat_stats)


# ---------- Основной сценарий ----------

def main():
    doc = Document()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(11)

    title = doc.add_heading("Отчёт по вариантам (листам) файла transactions.xlsx", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    sheets = pd.read_excel(INPUT_FILE, sheet_name=None, engine="openpyxl")
    sheet_names = list(sheets.keys())

    add_paragraph(doc, f"Файл: {INPUT_FILE}")
    add_paragraph(doc, f"Всего вариантов (листов): {len(sheet_names)}")
    add_paragraph(doc, "Список вариантов:")
    for name in sheet_names:
        add_paragraph(doc, f"  • {name} — {len(sheets[name])} строк")

    for name in sheet_names:
        solve_variant(doc, sheets[name].copy(), variant_name=name)

    doc.save(OUTPUT_FILE)
    print(f"Отчёт сохранён в '{OUTPUT_FILE}'")
    print(f"Обработано вариантов: {len(sheet_names)}")


if __name__ == "__main__":
    main()

Отчёт сохранён в 'report.docx'
Обработано вариантов: 10
